In [1]:
#premodel and seed generation implementation:

import numpy as np
import random

seed = 7
np.random.seed(seed) # for numpy
random.seed(seed) # for random and other libraries
print(f'Global Random seed set to: {seed}')

Global Random seed set to: 7


## Problem 1: Automatic Gift Recognizer
- uses data_proccesing
- uses model_builder
- uses model_trainer

In [2]:
from src.data_proccesing import explore_data, split_data_into_3sets
# loading and displaying data
dataset = np.load("data/dataset.npz")
X, y = dataset["X"], dataset["y"]
explore_data(X,y) # prints class distribution
X_train, y_train, X_val, y_val, X_test, y_test = split_data_into_3sets(X, y, randomseed = seed) #splits data into train, val and test
#showimage(X_train, 5) # shows first 5 images in training set

Shape of X: (13067, 400)
Shape of y: (13067,)
Number of classes: 15
Unique labels: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14]
    Class  Count
0       0    546
1       1    900
2       2    804
3       3    454
4       4   1022
5       5   1362
6       6    858
7       7    557
8       8    888
9       9    834
10     10    782
11     11   1601
12     12    702
13     13    884
14     14    873


**Our implementation with 3 k folds**

In [ ]:
from src.model_builder import buildmodel
from model_trainer_validator import train_eval_model_cv
from itertools import product # https://docs.python.org/3/library/itertools.html#itertools.product

# Source for sklearn model parameters:
# RandomForestClassifier: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html
# SVC: https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html
# LogisticRegression: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
# KNeighborsClassifier: https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html
#defining hyperparameter grids for each model:

rf_grid = {
    'n_estimators': [50, 100],
    'max_depth': [None, 10],
    'max_features': ['sqrt', 'log2']
}
svm_grid = {
    'C': [0.1, 10],
    'kernel': ['linear', 'rbf'],
}
lr_grid = {
    'C': [0.1, 10],
}
knn_grid = {
    'n_neighbors': [3, 7]
}

# loop through all of the model instances with different hyperparameters defined above
# use the model trainer and model selector here to find the best model and hyperparameters
model_grids = {
    'RF': rf_grid,
    'SVM': svm_grid,
    'LR': lr_grid,
    'KNN': knn_grid
}

best_individ_models = []

for model_name, param_grid in model_grids.items():
    for params in product(*param_grid.values()):
        kwargs = dict(zip(param_grid.keys(), params))
        model = buildmodel(model_name, **kwargs)
        score = train_eval_model_cv(model, X_train, y_train, X_val, y_val, cv=3, seed=seed)
        best_individ_models.append((model_name, kwargs, score))
    
#evaluate models and hyperparameters
best_individ_models.sort(key=lambda x: x[2], reverse=True)
print("\nTop 5 models and hyperparameters:")
for model_name, params, score in best_individ_models[:5]:
    print(f"Model: {model_name}, Params: {params}, CV Score: {score:.4f}")





Starting RandomForestClassifier(n_estimators=50, random_state=7) 3-fold cross-validation...


KeyboardInterrupt: 

SKLearns implementation of GridSearchCV

In [5]:
from sklearn.model_selection import GridSearchCV
from src.model_builder import buildmodel
'''
models_and_grids = {
    'RF': (buildmodel('RF'), {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20],
        'max_features': ['sqrt', 'log2']
    }),
    'SVM': (buildmodel('SVM'), {
        'C': [0.1, 1, 10],
        'kernel': ['linear', 'rbf']
    }),
    'LR': (buildmodel('LR'), {
        'C': [0.1, 1, 10],
        'solver': ['liblinear']
    }),
    'KNN': (buildmodel('KNN'), {
        'n_neighbors': [3, 5, 7]
    })
}
'''
models_and_grids = {
    'RF': (buildmodel('RF'), {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20],
        'max_features': ['sqrt', 'log2']
    }),
    'KNN': (buildmodel('KNN'), {
        'n_neighbors': [3, 5, 7]
    })
}


best_models = {}

for name, (model, param_grid) in models_and_grids.items():
    print(f"\nTuning {name}...")
    grid = GridSearchCV(model, param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=1)
    grid.fit(X_train, y_train)
    print(f"Best {name} params: {grid.best_params_}")
    print(f"Best {name} score: {grid.best_score_:.4f}")
    best_models[name] = grid.best_estimator_



Tuning RF...
Fitting 3 folds for each of 18 candidates, totalling 54 fits
Best RF params: {'max_depth': 20, 'max_features': 'sqrt', 'n_estimators': 200}
Best RF score: 0.7361

Tuning KNN...
Fitting 3 folds for each of 3 candidates, totalling 9 fits
Best KNN params: {'n_neighbors': 3}
Best KNN score: 0.6424


## Problem 2: PCAA analysis
- uses pcaaanalysis.py to visualize and analyze training data
- methodology: 
- ensure SEED and SOLVER are fixed
- Ensure each part of proccess where randomization can skew reproducibility is accounted for
- On the training split, create a cv with stratified k fold to account for randomization and 
- run GridSearchCV with a Pipeline([Scaler, PCA, Model]).
- PCA will be refit in each CV fold.
- Evaluate the tuned model(s) on the validation set to pick the best approach.
- Final training: refit the chosen pipeline (Scaler+PCA+Model with the chosen hyperparams) on train+validation.
- Final report: evaluate once on the test set.

In [4]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from src.model_builder import buildmodel_pca
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.base import clone

# global variables(SEED defined earlier)
SOLVER = 'full'
SEED = seed #for clarity

models_and_grids = {
    # SVM with PCA
    'SVM_PCA': (buildmodel_pca('SVM_PCA', randomseed=SEED, pca_solver=SOLVER), {
        'pca__n_components': [0.80, 0.90, 100],   # try different numbers of components with cumalative explained variance >= these numbers
        'clf__kernel': ['rbf'],
        'clf__C': [0.1, 1, 10],
        'clf__gamma': ['scale', 0.01, 0.001]
    }),
    # KNN with PCA
    'KNN_PCA': (buildmodel_pca('KNN_PCA', randomseed=SEED, pca_solver=SOLVER), {
        'pca__n_components': [0.80, 0.95, 64],
        'clf__n_neighbors': [3, 5, 7],
        'clf__weights': ['uniform', 'distance']
    }),
    #logistic regression with PCA
    'LR_PCA': (buildmodel_pca('LR_PCA', randomseed=SEED, pca_solver=SOLVER), {
        'pca__n_components': [0.80, 0.95, 64], #pass 64 sklearn keeps exactly 64 PCs
        'clf__C': [0.1, 1, 10],
        'clf__penalty': ['l2'],
        'clf__solver': ['lbfgs']
    }),
    # Random forest (no PCA)
    'RF': (buildmodel_pca('RF', randomseed= SEED), {
        'n_estimators': [100, 200],
        'max_depth': [None, 10, 20],
        'max_features': ['sqrt', 'log2']
    })
}

best_models_train = {}
val_results = {}
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED) #for reproducibility and stratified splits which is important for class-imbalanced data

for name, (model, param_grid) in models_and_grids.items():
    print(f"\nTuning {name}...")
    grid = GridSearchCV(model, param_grid, cv=cv, scoring='accuracy', n_jobs=-1, verbose=1) # performs the k-fold cross-validation
    # for each canadate pca_n_components and model hyperparameter combination
        #Fit StandardScaler on the train-fold, transform it.

        #Fit PCA on that transformed train-fold, transform it.

        #Fit clf on those PC scores.

        #Transform the val-fold with the same scaler/PCA and score it.
    grid.fit(X_train, y_train)  # <- PCA+scaler are fit only on training folds; this is a pipeline and calls pca.transform(scaled_X) before passing to model's .fit()
    print(f"Best {name} params: {grid.best_params_}")
    print(f"Best {name} CV score: {grid.best_score_:.4f}")
    best_models_train[name] = grid.best_estimator_ #GridSearchCV object with the best found model refit on full training set
    tuned = best_models_train[name]
    print('')
    # For PCA models, print actual k chosen and cumulative explained variance
    if name in ['SVM_PCA', 'LR_PCA', 'KNN_PCA']:
        print("PCA model detected, extracting PCA info...")
        pipe = best_models_train[name]
        pca    = pipe.named_steps['pca']
        k = getattr(pca, 'n_components_',None)  # resolved number of components chosen by PCA during fit
        evr_sum = pca.explained_variance_ratio_.sum()
        d_orig = X_train.shape[1]
        d_pc = pca.components_.shape[0] 
        print(f"\n{name}")
        print(f"  Original dims: {d_orig}")
        print(f"  Chosen k (resolved): {k}  (pca__n_components in grid may have been 0.xx or an int)")
        print(f"  Cumulative explained variance (TRAIN fit): {evr_sum:.4f}")

    # Validate on validation set
    print('')
    print("Validating on validation set...")
    y_val_pred = tuned.predict(X_val) #calls pca.transform(scaled_X_val) before passing to model's .predict()
    accuracy = accuracy_score(y_val, y_val_pred)
    f1w = f1_score(y_val, y_val_pred, average='weighted', zero_division=0)
    f1m = f1_score(y_val, y_val_pred, average='macro', zero_division=0)
    val_results[name] = {'accuracy': accuracy, 'f1_weighted': f1w, 'f1_macro': f1m}
    print(f"{name} on VALIDATION: {accuracy:.4f}, F1-weighted: {f1w:.4f}, F1-macro: {f1m:.4f}")
    print("-" * 40)

best_model = max(val_results, key=lambda k: val_results[k]['f1_macro']) #returns model name based on best f1_macro
print(f"\nBest model on validation set: {best_model} with f1_macro {val_results[best_model]['f1_macro']:.4f}")



print('tuning best model on val+train')
#refit best model on full training+validation set
winner = best_models_train[best_model] #actual model not string
X_train_val = np.vstack((X_train, X_val))
y_train_val = np.hstack((y_train, y_val))
final_model = clone(winner).fit(X_train_val, y_train_val)

#Test
print('Evaluating winner model on held-out test set...')
y_test_pred = final_model.predict(X_test)
test_acc = accuracy_score(y_test, y_test_pred)
test_f1m = f1_score(y_test, y_test_pred, average='macro', zero_division=0)
test_f1w = f1_score(y_test, y_test_pred, average='weighted', zero_division=0)

print('Final evaluation on TEST set:')
print(f"[TEST] {best_model}  acc={test_acc:.4f}, f1_weighted={test_f1w:.4f}, f1_macro={test_f1m:.4f}")
print(classification_report(y_test, y_test_pred, digits=3))
print(confusion_matrix(y_test, y_test_pred))


# === What each “Tuning …” block means ===
# 
# • “Fitting 3 folds …”:
#     GridSearchCV is running 3-fold cross-validation on the TRAIN split
#     for many hyperparameter combinations.
#
# • “Best … params”:
#     The hyperparameters that achieved the highest CV score (per your
#     scoring/refit setting) on the TRAIN folds.
#
# • “Best … CV score”:
#     The average CV score over the 3 folds for that best combo.
#     NOTE: this is NOT the held-out validation set.
#
# • PCA info (for *_PCA models):
#     - Original dims: number of raw features (e.g., 400 pixels).
#     - Chosen k (resolved): the actual number of principal components
#       PCA used after fitting. If you supplied a float (e.g., 0.8),
#       PCA picked the smallest k whose cumulative explained variance ≥ 80%.
#       (Example: k = 90 for 0.8.)
#     - Cumulative explained variance: total variance retained by those k
#       components (e.g., ~0.8015 = 80.15%).
#
# • “Validating on validation set …”:
#     Take the tuned pipeline (refit on the full TRAIN split) and evaluate
#     ONCE on the VALIDATION set. Report:
#       - Accuracy: overall correctness.
#       - F1-weighted: class F1s averaged with weights by class size
#         (tracks accuracy but considers precision/recall).
#       - F1-macro: unweighted mean of per-class F1s (each class counts
#         equally). Use this as the selection metric.






Tuning SVM_PCA...
Fitting 3 folds for each of 27 candidates, totalling 81 fits
Best SVM_PCA params: {'clf__C': 10, 'clf__gamma': 'scale', 'clf__kernel': 'rbf', 'pca__n_components': 0.8}
Best SVM_PCA CV score: 0.7805

PCA model detected, extracting PCA info...

SVM_PCA
  Original dims: 400
  Chosen k (resolved): 90  (pca__n_components in grid may have been 0.xx or an int)
  Cumulative explained variance (TRAIN fit): 0.8015

Validating on validation set...
SVM_PCA on VALIDATION: 0.7980, F1-weighted: 0.7963, F1-macro: 0.7872
----------------------------------------

Tuning KNN_PCA...
Fitting 3 folds for each of 18 candidates, totalling 54 fits
Best KNN_PCA params: {'clf__n_neighbors': 5, 'clf__weights': 'distance', 'pca__n_components': 64}
Best KNN_PCA CV score: 0.7563

PCA model detected, extracting PCA info...

KNN_PCA
  Original dims: 400
  Chosen k (resolved): 64  (pca__n_components in grid may have been 0.xx or an int)
  Cumulative explained variance (TRAIN fit): 0.7359

Validating 

## Problem 3: Bad Data